# Auditoría inicial del tamaño de los datos MiBici

Cuenta viajes y estaciones para el periodo acordado (octubre de 2025 a septiembre de 2026). La lectura se limita a las columnas de estación y no carga ni muestra identificadores de usuario, género, año de nacimiento ni registros individuales. Esta es una auditoría inicial de conteos, no una validación integral de calidad.

In [1]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
while not (project_root / "data" / "raw").is_dir() and project_root != project_root.parent:
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"
if not raw_data_dir.is_dir():
    raise FileNotFoundError("No se encontró data/raw desde la ubicación actual del notebook.")

months = pd.period_range(start="2025-10", end="2026-09", freq="M")
trip_files = [raw_data_dir / f"datos_abiertos_{month.strftime('%Y_%m')}.csv" for month in months]
missing_files = [path for path in trip_files if not path.is_file()]
if missing_files:
    missing_list = "\n".join(str(path) for path in missing_files)
    raise FileNotFoundError(f"Faltan archivos mensuales esperados:\n{missing_list}")

station_file = raw_data_dir / "nomenclatura_2026_09.csv"
if not station_file.is_file():
    raise FileNotFoundError(f"No se encontró el archivo de estaciones: {station_file}")

## Conteo de viajes y estaciones observadas en los viajes

Para cada CSV se leen únicamente `Origen_Id` y `Destino_Id`. El número de filas de esas columnas da el conteo de viajes leídos; los IDs únicos se agregan entre los meses para contar estaciones observadas en origen, destino y en ambos.

In [2]:
monthly_rows = []
origin_ids = set()
destination_ids = set()

for path, month in zip(trip_files, months, strict=True):
    trips = pd.read_csv(
        path,
        usecols=["Origen_Id", "Destino_Id"],
        dtype="string",
        encoding_errors="replace",
    )

    origins = trips["Origen_Id"].str.strip()
    destinations = trips["Destino_Id"].str.strip()
    origin_ids.update(origins.dropna().loc[lambda values: values.ne("")].unique())
    destination_ids.update(
        destinations.dropna().loc[lambda values: values.ne("")].unique()
    )
    monthly_rows.append({"mes": str(month), "viajes_leidos": len(trips)})

monthly_counts = pd.DataFrame(monthly_rows)
total_trips = int(monthly_counts["viajes_leidos"].sum())
trip_station_ids = origin_ids | destination_ids

print(f"Meses con archivo leído: {len(monthly_counts)}")
print(f"Viajes leídos en total: {total_trips:,}")
print(f"Estaciones únicas como origen: {len(origin_ids):,}")
print(f"Estaciones únicas como destino: {len(destination_ids):,}")
print(f"Estaciones únicas observadas (origen o destino): {len(trip_station_ids):,}")
monthly_counts

Meses con archivo leído: 12
Viajes leídos en total: 4,497,877
Estaciones únicas como origen: 471
Estaciones únicas como destino: 471
Estaciones únicas observadas (origen o destino): 471


,mes,viajes_leidos
0,2025-10,420064
1,2025-11,380328
2,2025-12,335420
3,2026-01,357180
4,2026-02,333756
5,2026-03,376243
6,2026-04,351851
7,2026-05,315895
8,2026-06,355303
9,2026-07,388257


## Conteo en la nomenclatura publicada

Se cuenta por separado el total de filas y los IDs distintos del archivo de nomenclatura. Este conteo no implica que todas las estaciones listadas estuvieran activas durante cada mes del periodo.

In [3]:
station_table = pd.read_csv(
    station_file,
    usecols=["id"],
    dtype="string",
    encoding_errors="replace",
)
station_ids = station_table["id"].str.strip().dropna()
station_ids = station_ids.loc[station_ids.ne("")]

station_summary = pd.DataFrame(
    [{
        "archivo": station_file.name,
        "filas_leidas": len(station_table),
        "ids_distintos": station_ids.nunique(),
        "ids_en_viajes_no_en_nomenclatura": len(trip_station_ids - set(station_ids)),
    }]
)
station_summary

,archivo,filas_leidas,ids_distintos,ids_en_viajes_no_en_nomenclatura
0,nomenclatura_2026_09.csv,484,484,2


## Interpretación y limitaciones

- Los conteos se calculan sobre las filas que pandas puede leer en los archivos presentes; no se han auditado duplicados, fechas inválidas ni otros errores de calidad.
- El conjunto de IDs observados en los viajes y el de la nomenclatura pueden diferir. La nomenclatura de septiembre de 2026 es una fotografía actual y su correspondencia con las estaciones históricas debe validarse antes de usarla como descripción del periodo completo.
- Las salidas son agregados. No se imprimen datos de viajes ni identificadores personales.